# Final Project: Multi-Output Side-Channel Attack Reproduction

This notebook reproduces the results from:
**"Efficient Nonprofiled Side-Channel Attack Using Multi-Output Classification Neural Network"**

## Overview

This implementation includes:
- MLP_MO: Multi-output MLP for masking and noise countermeasures
- CNN_MO: Multi-output CNN for de-synchronization countermeasures
- Evaluation and visualization tools

## Setup


In [21]:
import sys
from pathlib import Path

# Add src directory to Python path (one level up from notebooks/)
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_path = project_root / "src"

if src_path.exists():
    # Add both project_root and src to path for flexible imports
    sys.path.insert(0, str(src_path))
    sys.path.insert(0, str(project_root))
else:
    raise ImportError(f"Could not find src directory at: {src_path}")

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


Using device: cuda


## Import Modules


In [22]:
from data_preprocessing import (
    create_multi_output_labels,
    normalize_traces,
    load_ascad_dataset,
    load_chipwhisperer_dataset,
    create_noisy_dataset,
    create_desync_dataset,
    inject_gaussian_noise
)

from mlp_mo import MLP_MO, MultiOutputLoss
from cnn_mo import CNN_MO

from training import (
    PowerTraceDataset,
    TrainingLogger,
    train_mlp_mo,
    train_cnn_mo
)

from evaluation import evaluate_model, run_repeated_attacks
from visualization import (
    plot_accuracy_curves,
    plot_attack_time_comparison,
    plot_success_rate_comparison,
    plot_branch_accuracies
)


## Experiment 1: MLP_MO on Masking Countermeasure

Train MLP_MO (both Non-SoSL and SoSL-200) on ASCAD dataset with masking countermeasure.

**Note:** Uncomment code cells once datasets are available.


In [24]:
# Configuration
dataset_path = "../datasets/ASCAD/STM32_AES_v2/ascadv2-extracted.h5"
num_epochs = 50
batch_size = 32

# Load dataset
print("Loading ASCAD dataset...")
traces, plaintexts, correct_key = load_ascad_dataset(dataset_path)
print(f"Loaded {len(traces)} traces, trace length: {traces.shape[1]}")
print(f"Plaintexts shape: {plaintexts.shape}")
if correct_key is not None:
    print(f"Correct key: {correct_key}")
else:
    print("Warning: Correct key not found in dataset")

# Normalize traces
print("\nNormalizing traces...")
traces, _ = normalize_traces(traces, method='standard')
print(f"Traces normalized. Mean: {traces.mean():.4f}, Std: {traces.std():.4f}")

# Create multi-output labels
print("\nCreating multi-output labels...")
labels = create_multi_output_labels(plaintexts)
print(f"Labels shape: {labels.shape} (n_traces={labels.shape[0]}, n_key_hypotheses=256)")

# Split dataset (80% train, 20% validation)
print("\nSplitting dataset...")
n_train = int(0.8 * len(traces))
train_traces = traces[:n_train]
train_labels = labels[:n_train]
val_traces = traces[n_train:]
val_labels = labels[n_train:]
print(f"Training set: {len(train_traces)} traces")
print(f"Validation set: {len(val_traces)} traces")

# Create datasets and data loaders
train_dataset = PowerTraceDataset(train_traces, train_labels)
val_dataset = PowerTraceDataset(val_traces, val_labels)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("\n✓ Dataset preparation complete!")


Loading ASCAD dataset...


RuntimeError: Error loading ASCAD dataset from ../datasets/ASCAD/STM32_AES_v2/ascadv2-extracted.h5: Cannot compare structured or void to non-void arrays.

### Train Non-SoSL MLP_MO

Non-SoSL (no shared layer) variant for better discrimination.


In [ ]:
# Create Non-SoSL MLP_MO model
trace_length = traces.shape[1]
model_nonsol = MLP_MO(trace_length=trace_length, shared_layer_size=0)
print(f"Non-SoSL MLP_MO model created")
print(f"  Trace length: {trace_length}")
print(f"  Total parameters: {sum(p.numel() for p in model_nonsol.parameters()):,}")

# Create logger
logger_nonsol = TrainingLogger()

# Train model
print("\nStarting training...")
history_nonsol = train_mlp_mo(
    model_nonsol, train_loader, val_loader,
    num_epochs=num_epochs, device=device,
    logger=logger_nonsol, correct_key=correct_key
)

# Evaluate model
print("\nEvaluating model...")
metrics_nonsol = evaluate_model(model_nonsol, val_loader, device=device, correct_key=correct_key)

print(f"\n{'='*60}")
print("Non-SoSL Results:")
print(f"{'='*60}")
print(f"Attack time: {history_nonsol['attack_time']:.2f} seconds")
print(f"Mean accuracy: {metrics_nonsol['mean_accuracy']:.4f}")
if correct_key is not None:
    print(f"Correct key accuracy: {metrics_nonsol['correct_key_accuracy']:.4f}")
    print(f"Accuracy gap: {metrics_nonsol['accuracy_gap']:.4f}")
    print(f"Key ranking: {metrics_nonsol['key_ranking']}")
    print(f"Success: {metrics_nonsol['success']}")

# Save results
logger_nonsol.save("MLP_MO_NonSoSL_training.json")
print("\n✓ Training complete and results saved!")


### Train SoSL-200 MLP_MO

SoSL-200 (shared layer with 200 nodes) variant for better noise robustness.


In [ ]:
# Create SoSL-200 MLP_MO model
model_sol = MLP_MO(trace_length=trace_length, shared_layer_size=200)
print(f"SoSL-200 MLP_MO model created")
print(f"  Trace length: {trace_length}")
print(f"  Shared layer size: 200")
print(f"  Total parameters: {sum(p.numel() for p in model_sol.parameters()):,}")

# Create logger
logger_sol = TrainingLogger()

# Train model
print("\nStarting training...")
history_sol = train_mlp_mo(
    model_sol, train_loader, val_loader,
    num_epochs=num_epochs, device=device,
    logger=logger_sol, correct_key=correct_key
)

# Evaluate model
print("\nEvaluating model...")
metrics_sol = evaluate_model(model_sol, val_loader, device=device, correct_key=correct_key)

print(f"\n{'='*60}")
print("SoSL-200 Results:")
print(f"{'='*60}")
print(f"Attack time: {history_sol['attack_time']:.2f} seconds")
print(f"Mean accuracy: {metrics_sol['mean_accuracy']:.4f}")
if correct_key is not None:
    print(f"Correct key accuracy: {metrics_sol['correct_key_accuracy']:.4f}")
    print(f"Accuracy gap: {metrics_sol['accuracy_gap']:.4f}")
    print(f"Key ranking: {metrics_sol['key_ranking']}")
    print(f"Success: {metrics_sol['success']}")

# Save results
logger_sol.save("MLP_MO_SoSL200_training.json")
print("\n✓ Training complete and results saved!")


### Visualize Results


In [ ]:
# Plot accuracy curves
plot_accuracy_curves(history_nonsol, 
                    save_path="../figures/masking_MLP_MO_NonSoSL_accuracy.png",
                    title="MLP_MO Non-SoSL Training Accuracy")

plot_accuracy_curves(history_sol,
                    save_path="../figures/masking_MLP_MO_SoSL200_accuracy.png",
                    title="MLP_MO SoSL-200 Training Accuracy")

# Plot branch accuracies
if correct_key is not None:
    plot_branch_accuracies(metrics_nonsol['branch_accuracies'], correct_key=correct_key,
                          save_path="../figures/masking_MLP_MO_NonSoSL_branches.png",
                          title="Branch Accuracies - Non-SoSL")
    
    plot_branch_accuracies(metrics_sol['branch_accuracies'], correct_key=correct_key,
                          save_path="../figures/masking_MLP_MO_SoSL200_branches.png",
                          title="Branch Accuracies - SoSL-200")

# Compare attack times
attack_times = {
    'Non-SoSL': history_nonsol['attack_time'],
    'SoSL-200': history_sol['attack_time']
}
plot_attack_time_comparison(attack_times,
                           save_path="../figures/masking_attack_time_comparison.png",
                           title="Attack Time Comparison")

print("✓ All visualizations saved to ../figures/")


## Experiment 2: Noise Robustness

Evaluate MLP_MO on datasets with different noise levels (σ = 0.5, 1.0, 1.5).


In [ ]:
# Configuration for noise experiments
sigmas = [0.5, 1.0, 1.5]
num_attacks = 10  # Reduced for faster testing (paper uses 50)
num_epochs_noise = 30  # Reduced epochs for faster testing

# Reload original traces (before normalization) for noise injection
traces_original, plaintexts_original, _ = load_ascad_dataset(dataset_path)
traces_original, _ = normalize_traces(traces_original, method='standard')
labels_original = create_multi_output_labels(plaintexts_original)

noise_results = {}

for sigma in sigmas:
    print(f"\n{'='*60}")
    print(f"Testing noise level σ = {sigma}")
    print(f"{'='*60}\n")
    
    # Create noisy dataset
    noisy_traces = inject_gaussian_noise(traces_original, sigma=sigma)
    noisy_traces, _ = normalize_traces(noisy_traces, method='standard')
    
    # Split dataset
    n_train = int(0.8 * len(noisy_traces))
    train_traces_noise = noisy_traces[:n_train]
    train_labels_noise = labels_original[:n_train]
    val_traces_noise = noisy_traces[n_train:]
    val_labels_noise = labels_original[n_train:]
    
    train_dataset_noise = PowerTraceDataset(train_traces_noise, train_labels_noise)
    val_dataset_noise = PowerTraceDataset(val_traces_noise, val_labels_noise)
    train_loader_noise = torch.utils.data.DataLoader(train_dataset_noise, batch_size=batch_size, shuffle=True)
    val_loader_noise = torch.utils.data.DataLoader(val_dataset_noise, batch_size=batch_size, shuffle=False)
    
    # Run repeated attacks with SoSL-200 (better noise robustness)
    print(f"Running {num_attacks} attacks with SoSL-200...")
    model_kwargs = {'trace_length': trace_length, 'shared_layer_size': 200}
    
    repeated_results = run_repeated_attacks(
        MLP_MO, model_kwargs, train_loader_noise, val_loader_noise,
        num_attacks=num_attacks, num_epochs=num_epochs_noise,
        correct_key=correct_key, device=device
    )
    
    noise_results[f'σ={sigma}'] = repeated_results.get('success_rate', 0.0)
    
    print(f"\nResults for σ = {sigma}:")
    if correct_key is not None:
        print(f"  Success rate: {repeated_results['success_rate']*100:.1f}%")
    print(f"  Mean attack time: {repeated_results['mean_attack_time']:.2f} seconds")

# Plot results
if correct_key is not None:
    plot_success_rate_comparison(noise_results,
                                save_path="../figures/noise_success_rate.png",
                                title="Success Rate vs Noise Level")
    print("\n✓ Noise robustness experiment complete!")
else:
    print("\n⚠ Cannot compute success rate without correct key")


## Summary

### Expected Results (from paper)

- **MLP_MO Speedup**: 6-9x faster than DDLA baseline
- **Noise Robustness**: +20% success rate improvement at σ=1.0, 1.5
- **CNN_MO**: ~30x faster than CNN_DDLA on de-synchronized data (stretch goal)

### Notes

- **ChipWhisperer Dataset**: Moved to stretch goals (requires hardware setup)
- **De-synchronization Experiments**: Can be run if ChipWhisperer dataset becomes available
- **Baseline Comparison**: DDLA baseline implementation not included (compare with paper results)

### Generated Files

- Training logs: `results/training_logs/*.json`
- Figures: `figures/*.png`
